# Grupo 3 — validação sanitária da base de poluição

Este notebook reconstrói a data a partir de `year`, `month`, `day` e `hour`, verificando nulos, duplicatas e regularidade horária. A variável de interesse mais comum nesta base é `PM2.5`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'validacao_bases.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Não foi possível localizar validacao_bases.py.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from validacao_bases import (
    CONFIGURACOES, calcular_sha256, carregar_base, extrair_datas,
    relatorios_como_dataframe, validar_base,
)

NOME_BASE = 'poluicao'
config = CONFIGURACOES[NOME_BASE]
dados = carregar_base(NOME_BASE, RAIZ)
print(f'Base: {NOME_BASE} | formato: {dados.shape[0]:,} linhas x {dados.shape[1]} colunas')

Base: poluicao | formato: 35,064 linhas x 18 colunas


In [2]:
display(dados.head())
display(dados.dtypes.rename('tipo').to_frame())

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


,tipo
No,int64
year,int64
month,int64
day,int64
hour,int64
PM2.5,float64
PM10,float64
SO2,float64
NO2,float64
CO,float64


## Resultado consolidado

A frequência esperada é horária (`h`). A grade temporal pode estar completa mesmo que algumas variáveis medidas tenham valores nulos.

In [3]:
relatorio = validar_base(NOME_BASE, RAIZ)
display(relatorios_como_dataframe({NOME_BASE: relatorio}))
display(pd.Series(relatorio.nulos_por_coluna, name='quantidade_de_nulos').sort_values(ascending=False).to_frame())

,nome,linhas,colunas,linhas_com_nulos,duplicatas_exatas,datas_invalidas,datas_duplicadas,ordenacao_datas,frequencia_esperada,frequencia_regular,timestamps_ausentes,timestamps_fora_da_grade,aprovada
0,poluicao,35064,18,3249,0,0,0,crescente,h,True,0,0,False


,quantidade_de_nulos
CO,1776
O3,1719
NO2,1023
SO2,935
PM2.5,925
PM10,718
wd,81
TEMP,20
PRES,20
DEWP,20


In [4]:
datas = extrair_datas(dados, config)
problemas = dados.loc[dados.duplicated(keep=False) | datas.duplicated(keep=False)].copy()
problemas.insert(0, 'data_normalizada', datas.loc[problemas.index])
print(f'Linhas com duplicidade exata ou temporal: {len(problemas):,}')
display(problemas.head(10))
print('Exemplos de horas ausentes:', relatorio.exemplos_timestamps_ausentes)
print('Ordenação encontrada:', relatorio.ordenacao_datas)

Linhas com duplicidade exata ou temporal: 0


,data_normalizada,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station


Exemplos de horas ausentes: []
Ordenação encontrada: crescente


## Evidência para o congelamento

O hash identifica exatamente o arquivo analisado. O congelamento completo das cinco bases deve ser feito uma única vez com `congelar_bases('dados_congelados/v1')`.

In [5]:
arquivo = RAIZ / config.caminho
print('Arquivo:', arquivo.relative_to(RAIZ))
print('SHA-256:', calcular_sha256(arquivo))
print('Conclusão:', 'APROVADA' if relatorio.aprovada else 'REQUER TRATAMENTO ANTES DA MODELAGEM')

Arquivo: grupo3\PRSA_Data_Aotizhongxin_20130301-20170228.csv
SHA-256: a94fbcfc71708b6ffdc033360163d91efb7f675e496f6b5860929aa96b96351b
Conclusão: REQUER TRATAMENTO ANTES DA MODELAGEM
